In [ ]:
!pip -q install -U pip
!pip -q install -r requirements.txt
!pip -q install opencv-python tqdm
!pip -q install git+https://github.com/facebookresearch/segment-anything.git

In [ ]:
!rm -rf dinov2 dinov3
!git clone -q https://github.com/facebookresearch/dinov2.git
!git clone -q https://github.com/facebookresearch/dinov3.git

!pip -q install omegaconf torchmetrics fvcore iopath submitit ftfy regex scikit-learn termcolor

In [ ]:
import os
import shutil
import torch
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

MYDRIVE = '/content/drive/MyDrive'

DINOV2_WEIGHTS = os.path.join(MYDRIVE, 'dinov2_vitb14_pretrain_small.pth')
DINOV3_WEIGHTS = os.path.join(MYDRIVE, 'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth')
SAM_WEIGHTS_B  = os.path.join(MYDRIVE, 'sam_vit_b_01ec64_small.pth')

SPAIR_DIR = os.path.join(MYDRIVE, 'SPair-71k')
SPAIR_TAR = os.path.join(MYDRIVE, 'SPair-71k.tar.gz')
PFW_ZIP   = os.path.join(MYDRIVE, 'pf-willow.zip')

LOCAL_DATA_DIR = '/content/data'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

print('Drive paths:')
print(' - DINOV2_WEIGHTS:', DINOV2_WEIGHTS)
print(' - DINOV3_WEIGHTS:', DINOV3_WEIGHTS)
print(' - SAM_WEIGHTS_B :', SAM_WEIGHTS_B)
print(' - SPAIR_DIR     :', SPAIR_DIR)
print(' - SPAIR_TAR     :', SPAIR_TAR)
print(' - PFW_ZIP       :', PFW_ZIP)
print('Local data dir:', LOCAL_DATA_DIR)


In [ ]:
local_spair = os.path.join(LOCAL_DATA_DIR, 'SPair-71k')

if os.path.isdir(local_spair):
    print('SPair-71k already present at', local_spair)
elif os.path.isdir(SPAIR_DIR):
    print('Copying SPair-71k folder from Drive to local VM...')
    shutil.copytree(SPAIR_DIR, local_spair)
    print('Done:', local_spair)
elif os.path.isfile(SPAIR_TAR):
    print('Extracting SPair-71k.tar.gz from Drive to local VM...')
    shutil.unpack_archive(SPAIR_TAR, LOCAL_DATA_DIR, format='gztar')
    print('Done:', local_spair)
else:
    raise FileNotFoundError('Could not find SPair-71k folder or SPair-71k.tar.gz in MyDrive.')


In [ ]:
from segment_anything import SamPredictor, sam_model_registry

# SAM ViT-B
sam = sam_model_registry['vit_b'](checkpoint=SAM_WEIGHTS_B)
sam_predictor = SamPredictor(sam)

# DINOv2 and DINOv3 from local clones
# We load architecture from repo and then load weights from Drive.
dinov2 = torch.hub.load('dinov2', 'dinov2_vitb14', source='local', pretrained=False)
dinov2.load_state_dict(torch.load(DINOV2_WEIGHTS, map_location='cpu'))

dinov3 = torch.hub.load('dinov3', 'dinov3_vitb16', source='local', pretrained=False)
dinov3.load_state_dict(torch.load(DINOV3_WEIGHTS, map_location='cpu'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dinov2.to(device).eval()
dinov3.to(device).eval()
sam.to(device).eval()

print('Ready. Device:', device)
